In [1]:

import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import skfuzzy as fuzz
import skfuzzy.control as ctrl

In [7]:




st.set_page_config(
    page_title="SPK Performa Kapal – Fuzzy Mamdani",
    layout="wide",
)

# ───────────────────────────────────────────────────────────────────────────────
# LOAD DATA
# ───────────────────────────────────────────────────────────────────────────────
@st.cache_data
def load_data():
    df = pd.read_csv("Ship_Performance_Dataset.csv")
    # Menghapus baris yang kolom Ship_Type atau kriteria utamanya kosong (NaN)
    df = df.dropna(subset=["Ship_Type"] + KRITERIA)
    return df

df = load_data()

KRITERIA = [
    "Speed_Over_Ground_knots",
    "Efficiency_nm_per_kWh",
    "Revenue_per_Voyage_USD",
    "Operational_Cost_USD",
    "Average_Load_Percentage",
]

KRITERIA_LABEL = {
    "Speed_Over_Ground_knots":  "Kecepatan (knots)",
    "Efficiency_nm_per_kWh":    "Efisiensi (nm/kWh)",
    "Revenue_per_Voyage_USD":   "Pendapatan (USD)",
    "Operational_Cost_USD":     "Biaya Operasional (USD)",
    "Average_Load_Percentage":  "Rata-rata Muatan (%)",
}

BENEFIT = ["Speed_Over_Ground_knots", "Efficiency_nm_per_kWh",
           "Revenue_per_Voyage_USD", "Average_Load_Percentage"]
COST    = ["Operational_Cost_USD"]

# ───────────────────────────────────────────────────────────────────────────────
# SIDEBAR NAVIGATION
# ───────────────────────────────────────────────────────────────────────────────
with st.sidebar:
    st.image("https://img.icons8.com/color/96/cruise-ship.png", width=80)
    st.title("🚢 SPK Kapal")
    st.markdown("**Fuzzy Mamdani DSS**")
    st.markdown("---")
    halaman = st.radio(
        "Navigasi",
        ["📋 Dataset", "⚙️ Pengaturan Bobot", "🔢 Hitung SPK",
         "📊 Visualisasi", "👥 Profil Kelompok"],
    )
    st.markdown("---")
    st.caption("SCPK 2025/2026")

# ═══════════════════════════════════════════════════════════════════════════════
# HALAMAN 1 – DATASET
# ═══════════════════════════════════════════════════════════════════════════════
if halaman == "📋 Dataset":
    st.title("📋 Dataset Performa Kapal")
    st.markdown(
        f"Dataset memiliki **{len(df):,} baris** dan **{df.shape[1]} kolom**. "
        "Sumber: Ship Performance Dataset (Kaggle)."
    )

    col1, col2, col3 = st.columns(3)
    col1.metric("Total Data", f"{len(df):,}")
    col2.metric("Jumlah Kriteria (dipakai)", len(KRITERIA))
    col3.metric("Jenis Kapal", df["Ship_Type"].nunique())

    st.subheader("Tabel Dataset")
    filter_ship = st.multiselect(
        "Filter Jenis Kapal", options=sorted(df["Ship_Type"].unique()),
        default=sorted(df["Ship_Type"].unique())
    )
    df_view = df[df["Ship_Type"].isin(filter_ship)]
    st.dataframe(df_view.reset_index(drop=True), use_container_width=True, height=420)

    st.subheader("Statistik Deskriptif Kriteria")
    st.dataframe(df[KRITERIA].describe().rename(columns=KRITERIA_LABEL).T, use_container_width=True)

# ═══════════════════════════════════════════════════════════════════════════════
# HALAMAN 2 – PENGATURAN BOBOT
# ═══════════════════════════════════════════════════════════════════════════════
elif halaman == "⚙️ Pengaturan Bobot":
    st.title("⚙️ Pengaturan Bobot & Parameter Fuzzy")

    st.info(
        "Atur bobot kepentingan setiap kriteria (total tidak harus 100). "
        "Bobot akan dinormalisasi secara otomatis saat perhitungan."
    )

    st.subheader("🎚️ Bobot Kriteria")
    col1, col2 = st.columns(2)
    with col1:
        w1 = st.slider("Kecepatan (Benefit)",            0, 100, 25, key="w1")
        w2 = st.slider("Efisiensi (Benefit)",            0, 100, 25, key="w2")
        w3 = st.slider("Pendapatan (Benefit)",           0, 100, 20, key="w3")
    with col2:
        w4 = st.slider("Biaya Operasional (Cost)",       0, 100, 15, key="w4")
        w5 = st.slider("Rata-rata Muatan (Benefit)",     0, 100, 15, key="w5")

    bobot_raw = [w1, w2, w3, w4, w5]
    total = sum(bobot_raw)
    bobot_norm = [round(b / total, 4) for b in bobot_raw] if total > 0 else [0.2]*5

    st.markdown("**Bobot Ternormalisasi:**")
    df_bobot = pd.DataFrame({
        "Kriteria": list(KRITERIA_LABEL.values()),
        "Tipe": ["Benefit", "Benefit", "Benefit", "Cost", "Benefit"],
        "Bobot Input": bobot_raw,
        "Bobot Normal": bobot_norm,
    })
    st.dataframe(df_bobot, use_container_width=True, hide_index=True)

    st.subheader("📐 Parameter Fungsi Keanggotaan Fuzzy")
    st.markdown("Sesuaikan batas domain universe untuk setiap kriteria (opsional).")

    col_a, col_b = st.columns(2)
    with col_a:
        speed_max  = st.number_input("Kecepatan – batas atas (knots)", 15.0, 30.0, 25.0, 0.5)
        eff_max    = st.number_input("Efisiensi – batas atas (nm/kWh)", 0.1, 1.0, 0.5, 0.05)
        rev_max_m  = st.number_input("Pendapatan – batas atas (juta USD)", 0.5, 2.0, 1.0, 0.1)
    with col_b:
        cost_max_m = st.number_input("Biaya Operasional – batas atas (juta USD)", 0.1, 1.0, 0.5, 0.05)
        load_max   = st.number_input("Rata-rata Muatan – batas atas (%)", 80.0, 110.0, 100.0, 1.0)

    # simpan ke session state
    st.session_state["bobot_norm"] = bobot_norm
    st.session_state["params"] = {
        "speed_max": speed_max,
        "eff_max":   eff_max,
        "rev_max":   rev_max_m * 1_000_000,
        "cost_max":  cost_max_m * 1_000_000,
        "load_max":  load_max,
    }

    st.success("✅ Pengaturan tersimpan. Lanjut ke halaman **Hitung SPK**.")

# ═══════════════════════════════════════════════════════════════════════════════
# HALAMAN 3 – HITUNG SPK
# ═══════════════════════════════════════════════════════════════════════════════
elif halaman == "🔢 Hitung SPK":
    st.title("🔢 Perhitungan SPK – Fuzzy Mamdani")

    # ambil setting dari session state (atau default)
    bobot_norm = st.session_state.get("bobot_norm", [0.25, 0.25, 0.20, 0.15, 0.15])
    params     = st.session_state.get("params", {
        "speed_max": 25.0, "eff_max": 0.5,
        "rev_max": 1_000_000, "cost_max": 500_000, "load_max": 100.0
    })

    # ── pilihan filter ─────────────────────────────────────────────────────────
    col_f1, col_f2 = st.columns(2)
    with col_f1:
        filter_ship2 = st.multiselect(
            "Filter Jenis Kapal (opsional)",
            options=sorted(df["Ship_Type"].unique()),
            default=sorted(df["Ship_Type"].unique()),
        )
    with col_f2:
        n_top = st.selectbox("Tampilkan Top-N hasil", [10, 25, 50, 100, 250, 500], index=1)

    df_calc = df[df["Ship_Type"].isin(filter_ship2)][KRITERIA + ["Ship_Type"]].dropna().copy()
    st.write(f"Data yang diproses: **{len(df_calc):,} baris**")

    # ── TOMBOL EKSEKUSI ────────────────────────────────────────────────────────
    if st.button("🚀 Jalankan Perhitungan Fuzzy", type="primary", use_container_width=True):
        with st.spinner("Sedang menghitung… harap tunggu"):

            # ── 1. Normalisasi min-max ────────────────────────────────────────
            df_norm = df_calc[KRITERIA].copy()
            for k in BENEFIT:
                mn, mx = df_norm[k].min(), df_norm[k].max()
                df_norm[k] = (df_norm[k] - mn) / (mx - mn + 1e-9)
            for k in COST:
                mn, mx = df_norm[k].min(), df_norm[k].max()
                df_norm[k] = (mx - df_norm[k]) / (mx - mn + 1e-9)

            # ── 2. Definisi Universe & Fungsi Keanggotaan ─────────────────────
            # Universe output: skor 0-1
            x_out = np.linspace(0, 1, 200)

            # Universe input: setiap kriteria [0, 1] (sudah dinormalisasi)
            x_in = np.linspace(0, 1, 200)

            # Fungsi keanggotaan INPUT: Rendah, Sedang, Tinggi
            # trimf  = segitiga    (a, b, c)
            # trapmf = trapesium   (a, b, c, d)

            # Rendah – trapmf sisi kiri
            mf_rendah = fuzz.trapmf(x_in, [0, 0, 0.25, 0.5])
            # Sedang – trimf di tengah
            mf_sedang  = fuzz.trimf(x_in, [0.25, 0.5, 0.75])
            # Tinggi – trapmf sisi kanan
            mf_tinggi  = fuzz.trapmf(x_in, [0.5, 0.75, 1, 1])

            # Fungsi keanggotaan OUTPUT: Buruk, Cukup, Baik, Sangat Baik
            # Buruk – trapmf sisi kiri
            mf_buruk      = fuzz.trapmf(x_out, [0, 0, 0.2, 0.35])
            # Cukup – trimf
            mf_cukup      = fuzz.trimf(x_out, [0.25, 0.4, 0.55])
            # Baik – trimf
            mf_baik       = fuzz.trimf(x_out, [0.45, 0.6, 0.75])
            # Sangat Baik – trapmf sisi kanan
            mf_sgt_baik   = fuzz.trapmf(x_out, [0.65, 0.8, 1, 1])

            # ── 3. If-Then Rules & Inferensi per baris ────────────────────────
            scores = []
            for _, row in df_norm.iterrows():
                vals = row[KRITERIA].values  # 5 nilai [0,1]

                # Fuzzifikasi setiap kriteria
                mu_r = [fuzz.interp_membership(x_in, mf_rendah, v) for v in vals]
                mu_s = [fuzz.interp_membership(x_in, mf_sedang, v) for v in vals]
                mu_t = [fuzz.interp_membership(x_in, mf_tinggi, v) for v in vals]

                # Weighted derajat keanggotaan (bobot per kriteria)
                w = bobot_norm  # [w1..w5]

                score_r = sum(w[i] * mu_r[i] for i in range(5))
                score_s = sum(w[i] * mu_s[i] for i in range(5))
                score_t = sum(w[i] * mu_t[i] for i in range(5))

                # ── IF-THEN RULES (Mamdani) ────────────────────────────────
                # Rule 1: IF semua Rendah  → Buruk
                r1 = min(mu_r)
                # Rule 2: IF rata-rata Rendah → Cukup
                r2 = score_r
                # Rule 3: IF rata-rata Sedang → Baik
                r3 = score_s
                # Rule 4: IF semua Tinggi   → Sangat Baik
                r4 = min(mu_t)
                # Rule 5: IF kecepatan Tinggi AND efisiensi Tinggi → Sangat Baik
                r5 = min(mu_t[0], mu_t[1])
                # Rule 6: IF pendapatan Tinggi AND biaya Rendah   → Sangat Baik
                r6 = min(mu_t[2], mu_t[3])
                # Rule 7: IF muatan Tinggi AND kecepatan Sedang   → Baik
                r7 = min(mu_t[4], mu_s[0])
                # Rule 8: IF kecepatan Rendah AND biaya Tinggi    → Buruk
                r8 = min(mu_r[0], mu_t[3])
                # Rule 9: IF efisiensi Rendah AND pendapatan Rendah → Cukup
                r9 = min(mu_r[1], mu_r[2])

                # Agregasi output (clipping / min)
                agg_buruk    = np.fmin(max(r1, r8),       mf_buruk)
                agg_cukup    = np.fmin(max(r2, r9),       mf_cukup)
                agg_baik     = np.fmin(max(r3, r7),       mf_baik)
                agg_sgt_baik = np.fmin(max(r4, r5, r6),  mf_sgt_baik)

                agregasi = np.fmax(np.fmax(agg_buruk, agg_cukup),
                                   np.fmax(agg_baik,  agg_sgt_baik))

                # Defuzzifikasi – Centroid
                skor = fuzz.defuzz(x_out, agregasi, "centroid")
                scores.append(round(skor, 6))

            df_calc = df_calc.copy()
            df_calc["Skor_Fuzzy"] = scores
            df_result = df_calc.sort_values("Skor_Fuzzy", ascending=False).reset_index(drop=True)
            df_result.index += 1
            df_result.index.name = "Peringkat"
            df_result = df_result.rename(columns=KRITERIA_LABEL)

            st.session_state["df_result"]  = df_result
            st.session_state["x_out"]      = x_out
            st.session_state["mf_data"]    = {
                "buruk": mf_buruk, "cukup": mf_cukup,
                "baik": mf_baik, "sgt_baik": mf_sgt_baik,
            }
            st.session_state["mf_input"]   = {
                "x_in": x_in,
                "rendah": mf_rendah, "sedang": mf_sedang, "tinggi": mf_tinggi,
            }

        st.success("✅ Perhitungan selesai!")

    # ── Tampilkan hasil ────────────────────────────────────────────────────────
    if "df_result" in st.session_state:
        df_result = st.session_state["df_result"]

        # ── Tampilan kurva keanggotaan ─────────────────────────────────────
        st.subheader("📐 Kurva Keanggotaan Output (Mamdani)")
        x_out  = st.session_state["x_out"]
        mf_d   = st.session_state["mf_data"]
        mf_inp = st.session_state["mf_input"]

        fig_mf, axes = plt.subplots(1, 2, figsize=(13, 4))

        # Input MF
        ax0 = axes[0]
        ax0.plot(mf_inp["x_in"], mf_inp["rendah"], "r-",  lw=2, label="Rendah (trapmf)")
        ax0.plot(mf_inp["x_in"], mf_inp["sedang"], "y-",  lw=2, label="Sedang (trimf)")
        ax0.plot(mf_inp["x_in"], mf_inp["tinggi"], "g-",  lw=2, label="Tinggi (trapmf)")
        ax0.fill_between(mf_inp["x_in"], mf_inp["rendah"], alpha=0.15, color="red")
        ax0.fill_between(mf_inp["x_in"], mf_inp["sedang"], alpha=0.15, color="gold")
        ax0.fill_between(mf_inp["x_in"], mf_inp["tinggi"], alpha=0.15, color="green")
        ax0.set_title("Fungsi Keanggotaan Input (Normalisasi)")
        ax0.set_xlabel("Nilai Ternormalisasi [0, 1]")
        ax0.set_ylabel("Derajat Keanggotaan")
        ax0.legend(fontsize=8); ax0.grid(alpha=0.3)

        # Output MF
        ax1 = axes[1]
        ax1.plot(x_out, mf_d["buruk"],    "r-",  lw=2, label="Buruk (trapmf)")
        ax1.plot(x_out, mf_d["cukup"],    "y-",  lw=2, label="Cukup (trimf)")
        ax1.plot(x_out, mf_d["baik"],     "b-",  lw=2, label="Baik (trimf)")
        ax1.plot(x_out, mf_d["sgt_baik"], "g-",  lw=2, label="Sangat Baik (trapmf)")
        ax1.fill_between(x_out, mf_d["buruk"],    alpha=0.12, color="red")
        ax1.fill_between(x_out, mf_d["cukup"],    alpha=0.12, color="gold")
        ax1.fill_between(x_out, mf_d["baik"],     alpha=0.12, color="blue")
        ax1.fill_between(x_out, mf_d["sgt_baik"], alpha=0.12, color="green")
        ax1.set_title("Fungsi Keanggotaan Output (Skor Performa)")
        ax1.set_xlabel("Skor Fuzzy [0, 1]")
        ax1.set_ylabel("Derajat Keanggotaan")
        ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

        plt.tight_layout()
        st.pyplot(fig_mf)
        plt.close()

        # ── Tabel If-Then Rules ────────────────────────────────────────────
        st.subheader("📜 If-Then Rules (Mamdani)")
        rules_df = pd.DataFrame({
            "No.": range(1, 10),
            "Rule (IF → THEN)": [
                "IF semua kriteria Rendah → Buruk",
                "IF agregat bobot Rendah → Cukup",
                "IF agregat bobot Sedang → Baik",
                "IF semua kriteria Tinggi → Sangat Baik",
                "IF Kecepatan Tinggi AND Efisiensi Tinggi → Sangat Baik",
                "IF Pendapatan Tinggi AND Biaya Rendah → Sangat Baik",
                "IF Muatan Tinggi AND Kecepatan Sedang → Baik",
                "IF Kecepatan Rendah AND Biaya Tinggi → Buruk",
                "IF Efisiensi Rendah AND Pendapatan Rendah → Cukup",
            ],
            "Operator": ["min", "Σw·μ", "Σw·μ", "min", "min", "min", "min", "min", "min"],
        })
        st.dataframe(rules_df, use_container_width=True, hide_index=True)

        # ── Tabel Defuzzifikasi (sampel) ───────────────────────────────────
        st.subheader("📊 Tabel Defuzzifikasi (Metode: Centroid)")
        st.markdown(
            "Berikut adalah contoh nilai skor hasil defuzzifikasi centroid "
            "dari **10 data teratas**:"
        )
        st.dataframe(
            df_result[["Ship_Type", "Skor_Fuzzy"]].head(10),
            use_container_width=True,
        )

        # ── Tabel Hasil Perangkingan ──────────────────────────────────────
        st.subheader(f"🏆 Hasil Perangkingan (Top {n_top})")
        st.dataframe(
            df_result.head(n_top),
            use_container_width=True,
            height=480,
        )

        # ── Download ──────────────────────────────────────────────────────
        csv_out = df_result.to_csv().encode("utf-8")
        st.download_button(
            "⬇️ Download Hasil sebagai CSV",
            data=csv_out, file_name="hasil_spk_fuzzy.csv", mime="text/csv",
        )

# ═══════════════════════════════════════════════════════════════════════════════
# HALAMAN 4 – VISUALISASI
# ═══════════════════════════════════════════════════════════════════════════════
elif halaman == "📊 Visualisasi":
    st.title("📊 Visualisasi Data Analitik")

    if "df_result" not in st.session_state:
        st.warning("⚠️ Jalankan perhitungan SPK terlebih dahulu di halaman **Hitung SPK**.")
        st.stop()

    df_result = st.session_state["df_result"]
    df_viz    = df_result.copy()

    # ── Grafik 1: Distribusi Skor Fuzzy ───────────────────────────────────────
    st.subheader("1. Distribusi Skor Fuzzy Seluruh Kapal")
    fig1, ax1 = plt.subplots(figsize=(10, 4))
    ax1.hist(df_viz["Skor_Fuzzy"], bins=40, color="#3B82F6", edgecolor="white", alpha=0.85)
    ax1.axvline(df_viz["Skor_Fuzzy"].mean(), color="red", lw=2, linestyle="--",
                label=f"Rata-rata: {df_viz['Skor_Fuzzy'].mean():.3f}")
    ax1.set_xlabel("Skor Fuzzy"); ax1.set_ylabel("Frekuensi")
    ax1.set_title("Histogram Distribusi Skor Fuzzy Performa Kapal")
    ax1.legend(); ax1.grid(alpha=0.3)
    st.pyplot(fig1); plt.close()

    # ── Grafik 2: Rata-rata Skor per Jenis Kapal ──────────────────────────────
    st.subheader("2. Rata-rata Skor Fuzzy per Jenis Kapal")
    avg_per_type = df_viz.groupby("Ship_Type")["Skor_Fuzzy"].mean().sort_values(ascending=False)
    fig2, ax2 = plt.subplots(figsize=(10, 4))
    colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(avg_per_type)))[::-1]
    bars = ax2.bar(avg_per_type.index, avg_per_type.values, color=colors, edgecolor="white")
    ax2.bar_label(bars, fmt="%.3f", fontsize=9, padding=3)
    ax2.set_ylabel("Rata-rata Skor Fuzzy")
    ax2.set_title("Perbandingan Performa Rata-rata Antar Jenis Kapal")
    ax2.set_ylim(0, avg_per_type.max() * 1.15)
    plt.xticks(rotation=20, ha="right")
    ax2.grid(axis="y", alpha=0.3)
    st.pyplot(fig2); plt.close()

    # ── Grafik 3: Top-10 Kapal Terbaik ────────────────────────────────────────
    st.subheader("3. Top-10 Kapal Terbaik (Skor Fuzzy)")
    top10 = df_viz.head(10).copy()
    top10["Label"] = [f"#{i} – {t}" for i, t in
                      zip(top10.index, top10["Ship_Type"])]
    fig3, ax3 = plt.subplots(figsize=(10, 5))
    bar_colors = plt.cm.RdYlGn(np.linspace(0.35, 0.9, 10))[::-1]
    ax3.barh(top10["Label"][::-1], top10["Skor_Fuzzy"][::-1], color=bar_colors[::-1])
    ax3.set_xlabel("Skor Fuzzy")
    ax3.set_title("Top-10 Kapal dengan Performa Terbaik")
    ax3.set_xlim(0, 1)
    ax3.grid(axis="x", alpha=0.3)
    st.pyplot(fig3); plt.close()

    # ── Grafik 4: Scatter Kecepatan vs Efisiensi ──────────────────────────────
    st.subheader("4. Scatter: Kecepatan vs Efisiensi (warna = Skor Fuzzy)")
    col_spd = KRITERIA_LABEL["Speed_Over_Ground_knots"]
    col_eff = KRITERIA_LABEL["Efficiency_nm_per_kWh"]
    fig4, ax4 = plt.subplots(figsize=(10, 5))
    sc = ax4.scatter(
        df_viz[col_spd], df_viz[col_eff],
        c=df_viz["Skor_Fuzzy"], cmap="RdYlGn", alpha=0.6, s=15
    )
    plt.colorbar(sc, ax=ax4, label="Skor Fuzzy")
    ax4.set_xlabel("Kecepatan (knots)"); ax4.set_ylabel("Efisiensi (nm/kWh)")
    ax4.set_title("Scatter Plot: Kecepatan vs Efisiensi Kapal")
    ax4.grid(alpha=0.3)
    st.pyplot(fig4); plt.close()

    # ── Grafik 5: Boxplot Skor per Jenis Kapal ────────────────────────────────
    st.subheader("5. Distribusi Skor Fuzzy per Jenis Kapal (Boxplot)")
    ship_types = sorted(df_viz["Ship_Type"].unique())
    data_box   = [df_viz[df_viz["Ship_Type"] == t]["Skor_Fuzzy"].values for t in ship_types]
    fig5, ax5  = plt.subplots(figsize=(10, 5))
    bp = ax5.boxplot(data_box, labels=ship_types, patch_artist=True, notch=False)
    colors_box = plt.cm.Set2(np.linspace(0, 1, len(ship_types)))
    for patch, color in zip(bp["boxes"], colors_box):
        patch.set_facecolor(color)
    ax5.set_ylabel("Skor Fuzzy"); plt.xticks(rotation=20, ha="right")
    ax5.set_title("Boxplot Distribusi Skor Fuzzy per Jenis Kapal")
    ax5.grid(axis="y", alpha=0.3)
    st.pyplot(fig5); plt.close()

# ═══════════════════════════════════════════════════════════════════════════════
# HALAMAN 5 – PROFIL KELOMPOK
# ═══════════════════════════════════════════════════════════════════════════════
elif halaman == "👥 Profil Kelompok":
    st.title("👥 Profil Kelompok")

    col1, col2 = st.columns(2)

    with col1:
        st.markdown("""
        ### 🧑‍🎓 Anggota Kelompok

        | NIM | Nama |
        |-----|------|
        | **123240101** | Kurnia Ardiningrum |
        | **123240196** | Sabrina Alya |

        ---
        ### 📚 Mata Kuliah
        **Sistem Pendukung Keputusan (SCPK)**
        Tahun Ajaran 2025/2026
        """)

    with col2:
        st.markdown("""
        ### 🚢 Tentang Proyek

        **Judul:** Sistem Pendukung Keputusan Penilaian Performa Kapal
        Menggunakan Metode Fuzzy Mamdani

        **Dataset:** Ship Performance Dataset (Kaggle)
        - 2.736 baris data
        - 18 kolom fitur
        - 5 kriteria digunakan

        **Metode:** Fuzzy Mamdani
        - Fungsi keanggotaan: `trimf` & `trapmf`
        - 9 If-Then Rules
        - Defuzzifikasi: Centroid

        ---
        ### 🎯 Kriteria yang Digunakan

        | Kriteria | Tipe |
        |----------|------|
        | Kecepatan (knots) | Benefit |
        | Efisiensi (nm/kWh) | Benefit |
        | Pendapatan (USD) | Benefit |
        | Biaya Operasional (USD) | Cost |
        | Rata-rata Muatan (%) | Benefit |
        """)

    st.markdown("---")
    st.info("Proyek ini dibuat sebagai pemenuhan Tugas Akhir Praktikum SCPK 2025/2026.")


2026-06-02 22:56:06.078 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 22:56:06.078 No runtime found, using MemoryCacheStorageManager
2026-06-02 22:56:06.078 No runtime found, using MemoryCacheStorageManager
2026-06-02 22:56:06.078 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 22:56:06.100 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 22:56:06.101 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 22:56:06.102 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 22:56:06.102 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-02 22:56:06.102 Thread 'MainThread': missing ScriptRunContext! This warning can be ignor